### Extract & Load Raw Data

In [1]:
import pandas as pd
import requests
import sqlite3
import os

print("--- [ELT PHASE: EXTRACT & LOAD] ---")

# 1. Membuat direktori untuk menyimpan database ELT
os.makedirs('data_warehouse_elt', exist_ok=True)
db_path_elt = 'data_warehouse_elt/airbnb_elt_warehouse.db'

# ==========================================
# E - EXTRACT: Mengambil Data Mentah
# ==========================================
print("\n🔄 Langkah 1: Mengekstrak data mentah asli...")

file_path = '/content/AB_US_2023.csv'
if not os.path.exists(file_path):
    print(f"❌ ERROR: File '{file_path}' tidak ditemukan!")
    print("💡 Solusi: Pastikan file AB_US_2023.csv sudah di-upload ke Colab ini.")
    df_airbnb_raw = pd.DataFrame()
else:
    df_airbnb_raw = pd.read_csv(file_path, low_memory=False)
    print(f"✅ Airbnb Mentah Berhasil Diekstrak: {df_airbnb_raw.shape[0]} baris.")

# Ekstraksi data sekunder via API
api_url = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/geonames-all-cities-with-a-population-1000/records?where=cou_name_en%3D%22United%20States%22&limit=100"
try:
    response = requests.get(api_url, timeout=10)
    if response.status_code == 200:
        df_demo_raw = pd.DataFrame(response.json()['results'])
        print(f"✅ Demografi Mentah Berhasil Diekstrak: {df_demo_raw.shape[0]} baris.")
    else:
        df_demo_raw = pd.DataFrame()
        print("⚠ API gagal merespon. Menggunakan data kosong untuk fallback.")
except Exception as e:
    df_demo_raw = pd.DataFrame()
    print(f"⚠ Koneksi API terputus ({e}). Menggunakan data kosong untuk fallback.")


# ==========================================
# L - LOAD: Langsung Masuk Database Apa Adanya
# ==========================================
print("\n🔄 Langkah 2: Memuat (Load) data mentah langsung ke Database SQL...")

# Membuka koneksi ke database gudang data ELT
conn_elt = sqlite3.connect(db_path_elt)

# Memasukkan data Airbnb mentah ke SQL
if not df_airbnb_raw.empty:
    df_airbnb_raw.to_sql('raw_airbnb', conn_elt, if_exists='replace', index=False)
    print("➡ Tabel 'raw_airbnb' (MENTAH) berhasil diciptakan di DB.")

# Memasukkan data Demografi mentah ke SQL dengan perbaikan tipe data list/dict
if not df_demo_raw.empty:
    # PERBAIKAN MUTLAK: Mengonversi elemen list/dict menjadi string agar SQLite tidak error
    for col in df_demo_raw.columns:
        df_demo_raw[col] = df_demo_raw[col].apply(lambda x: str(x) if isinstance(x, (list, dict)) else x)

    df_demo_raw.to_sql('raw_demografi', conn_elt, if_exists='replace', index=False)
    print("➡ Tabel 'raw_demografi' (MENTAH) berhasil diciptakan di DB.")

# Menutup koneksi database sementara
conn_elt.close()

print(f"\n🎉 TAHAP E & L SELESAI SEMPURNA!")
print(f"Data mentah Anda kini sudah aman tersimpan di database: '{db_path_elt}'")

--- [ELT PHASE: EXTRACT & LOAD] ---

🔄 Langkah 1: Mengekstrak data mentah asli...
✅ Airbnb Mentah Berhasil Diekstrak: 232147 baris.
✅ Demografi Mentah Berhasil Diekstrak: 100 baris.

🔄 Langkah 2: Memuat (Load) data mentah langsung ke Database SQL...
➡ Tabel 'raw_airbnb' (MENTAH) berhasil diciptakan di DB.
➡ Tabel 'raw_demografi' (MENTAH) berhasil diciptakan di DB.

🎉 TAHAP E & L SELESAI SEMPURNA!
Data mentah Anda kini sudah aman tersimpan di database: 'data_warehouse_elt/airbnb_elt_warehouse.db'


### Transformasi Data dan Star Schema

In [2]:
import sqlite3

print("--- [ELT PHASE: TRANSFORM VIA SQL] ---")

db_path_elt = 'data_warehouse_elt/airbnb_elt_warehouse.db'
conn_elt = sqlite3.connect(db_path_elt)
cursor = conn_elt.cursor()

# 1. Menghitung batas IQR untuk Outlier Harga langsung dari database mentah
cursor.execute("""
    WITH ordered_price AS (
        SELECT price, ROW_NUMBER() OVER (ORDER BY price) as row_num, COUNT(*) OVER () as total_rows
        FROM raw_airbnb
        WHERE price IS NOT NULL
    )
    SELECT
        (SELECT price FROM ordered_price WHERE row_num = CAST(total_rows * 0.25 AS INT)) as q1,
        (SELECT price FROM ordered_price WHERE row_num = CAST(total_rows * 0.75 AS INT)) as q3
""")
q1, q3 = cursor.fetchone()
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"📊 Batas IQR Harga Terhitung -> Batas Bawah: {lower_bound}, Batas Atas: {upper_bound}")

# =========================================================================
# 2. PROSES TRANSFORMASI BESAR: MEMBENTUK STRUKTUR STAR SCHEMA DENGAN SQL
# =========================================================================
print("\n🛠 Menjalankan Query SQL Transformasi...")

# --- A. Membuat Tabel Dimensi: dim_host ---
cursor.execute("DROP TABLE IF EXISTS dim_host;")
cursor.execute(f"""
    CREATE TABLE dim_host AS
    SELECT DISTINCT
        host_id,
        COALESCE(host_name, 'Unknown') as host_name,
        calculated_host_listings_count,
        CASE WHEN calculated_host_listings_count >= 5 THEN 'Dominant' ELSE 'Normal' END as host_dominance
    FROM raw_airbnb
    GROUP BY host_id;
""")
print("✅ Dimensi 'dim_host' (Profil Host & Fitur 5) Berhasil Dibuat.")

# --- B. Membuat Tabel Dimensi: dim_city ---
cursor.execute("DROP TABLE IF EXISTS dim_city;")
# PERBAIKAN MUTLAK: Mengganti kolom 'city' menjadi 'name' sesuai struktur asli API mentah
cursor.execute("""
    CREATE TABLE dim_city AS
    SELECT DISTINCT
        UPPER(SUBSTR(name, 1, 1)) || LOWER(SUBSTR(name, 2)) as city,
        population as populasi_kota_api,
        timezone
    FROM raw_demografi
    WHERE name IS NOT NULL
    GROUP BY name;
""")
print("✅ Dimensi 'dim_city' (Geografis & Data API Enrichment) Berhasil Dibuat.")

# --- C. Membuat Tabel Dimensi: dim_time ---
cursor.execute("DROP TABLE IF EXISTS dim_time;")
cursor.execute("""
    CREATE TABLE dim_time AS
    SELECT DISTINCT
        last_review,
        CAST(SUBSTR(last_review, 1, 4) AS INTEGER) as year,
        CAST(SUBSTR(last_review, 6, 2) AS INTEGER) as month,
        CAST(SUBSTR(last_review, 9, 2) AS INTEGER) as day
    FROM raw_airbnb
    WHERE last_review IS NOT NULL AND last_review != '';
""")
print("✅ Dimensi 'dim_time' (Waktu Tren) Berhasil Dibuat.")

# --- D. Menghitung nilai Min-Max Harga untuk normalisasi di Tabel Fakta ---
cursor.execute(f"SELECT MIN(price), MAX(price) FROM raw_airbnb WHERE price >= {lower_bound} AND price <= {upper_bound}")
min_p, max_p = cursor.fetchone()

# --- E. Membuat Tabel Fakta Utama: fact_airbnb (Pusat Analisis & Validasi) ---
cursor.execute("DROP TABLE IF EXISTS fact_airbnb;")
cursor.execute(f"""
    CREATE TABLE fact_airbnb AS
    SELECT
        id,
        host_id,
        UPPER(SUBSTR(city, 1, 1)) || LOWER(SUBSTR(city, 2)) as city,
        last_review,
        room_type,
        price,
        minimum_nights,
        number_of_reviews,

        -- [Fitur 1: price_category] (Pengelompokan manual via SQL logika kelas ekonomi)
        CASE WHEN price <= 80 THEN 'Murah'
             WHEN price <= 180 THEN 'Sedang'
             ELSE 'Mahal' END as price_category,

        -- [Fitur 2: review_engagement]
        CASE WHEN number_of_reviews > 50 THEN 'Tinggi' ELSE 'Rendah' END as review_engagement,

        -- [Fitur 3: is_highly_available]
        CASE WHEN availability_365 >= 180 THEN 1 ELSE 0 END as is_highly_available,

        -- [Fitur 4: total_estimated_revenue]
        (price * minimum_nights) as total_estimated_revenue,

        -- [Normalisasi Numerik SQL - Bab 6.2.b] (Skala harga 0 sampai 1)
        ROUND(CAST(price - {min_p} AS FLOAT) / ({max_p} - {min_p}), 4) as normalized_price,

        -- [Encoding Kategorikal SQL - Bab 6.2.b] (Mengubah teks kamar menjadi ID angka)
        CASE WHEN room_type = 'Entire home/apt' THEN 0
             WHEN room_type = 'Hotel room' THEN 1
             WHEN room_type = 'Private room' THEN 2
             ELSE 3 END as room_type_encoded

    FROM raw_airbnb
    WHERE price >= {lower_bound} AND price <= {upper_bound} -- Eliminasi Outlier otomatis
      AND id IS NOT NULL
    GROUP BY id; -- Menjamin Aturan 1 Kelulusan Unik ID
""")
print("✅ Tabel Fakta 'fact_airbnb' (Transaksi & Fitur 1-4, Normalisasi, Encoding) Berhasil Dibuat.")

conn_elt.commit()
conn_elt.close()

print("\n🎉 SELURUH PROSES TRANSFORMATION (T) DI DALAM DATABASE SELESAI!")

--- [ELT PHASE: TRANSFORM VIA SQL] ---
📊 Batas IQR Harga Terhitung -> Batas Bawah: -147.5, Batas Atas: 488.5

🛠 Menjalankan Query SQL Transformasi...
✅ Dimensi 'dim_host' (Profil Host & Fitur 5) Berhasil Dibuat.
✅ Dimensi 'dim_city' (Geografis & Data API Enrichment) Berhasil Dibuat.
✅ Dimensi 'dim_time' (Waktu Tren) Berhasil Dibuat.
✅ Tabel Fakta 'fact_airbnb' (Transaksi & Fitur 1-4, Normalisasi, Encoding) Berhasil Dibuat.

🎉 SELURUH PROSES TRANSFORMATION (T) DI DALAM DATABASE SELESAI!


### Validasi Kualitas & SQL Analitik

In [3]:
import sqlite3
import pandas as pd

print("--- [ELT PHASE: VALIDATION & ANALYTICS VIA SQL] ---")

db_path_elt = 'data_warehouse_elt/airbnb_elt_warehouse.db'
conn_elt = sqlite3.connect(db_path_elt)

# Fungsi pembantu untuk menjalankan query analitik dengan rapi
def jalankan_analisis_sql(nomor_query, deskripsi, query_string):
    print(f"\n=======================================================")
    print(f"📊 QUERY {nomor_query}: {deskripsi}")
    print(f"=======================================================")
    df_hasil = pd.read_sql_query(query_string, conn_elt)
    display(df_hasil)

# =============================================================
# PART A: VALIDASI KUALITAS DATA WAREHOUSE (6 ATURAN WAJIB)
# =============================================================
print("\n🔍 Memulai 6 Aturan Uji Validasi Kualitas Data Warehouse...")
failed_rules = 0

# Aturan 1: ID Transaksi Unik
res1 = pd.read_sql_query("SELECT COUNT(id) - COUNT(DISTINCT id) as duplikat FROM fact_airbnb;", conn_elt)
if res1['duplikat'].iloc[0] > 0:
    print("❌ Aturan 1 Gagal: Ditemukan duplikasi ID di fact_airbnb.")
    failed_rules += 1
else:
    print("✅ Aturan 1 Lulus: Seluruh ID transaksi 100% unik.")

# Aturan 2: Nilai Harga Positif
res2 = pd.read_sql_query("SELECT COUNT(*) as cacat FROM fact_airbnb WHERE price < 0;", conn_elt)
if res2['cacat'].iloc[0] > 0:
    print("❌ Aturan 2 Gagal: Terdeteksi harga bernilai negatif.")
    failed_rules += 1
else:
    print("✅ Aturan 2 Lulus: Logika harga valid (tidak ada minus).")

# Aturan 3: Konsistensi tipe data minimum_nights (Deteksi teks ilegal)
res3 = pd.read_sql_query("SELECT COUNT(*) as cacat FROM fact_airbnb WHERE TYPEOF(minimum_nights) NOT IN ('integer', 'real');", conn_elt)
if res3['cacat'].iloc[0] > 0:
    print("❌ Aturan 3 Gagal: Terdapat tipe data non-numerik pada minimum_nights.")
    failed_rules += 1
else:
    print("✅ Aturan 3 Lulus: Tipe data kolom minimum_nights konsisten berupa angka.")

# Aturan 4: Logika ketersediaan tahunan
res4 = pd.read_sql_query("SELECT COUNT(*) as cacat FROM raw_airbnb WHERE availability_365 > 365;", conn_elt)
# Catatan: Kita cek di tabel raw karena di tabel fakta kolom ini direkayasa menjadi flag is_highly_available
if res4['cacat'].iloc[0] > 0:
    print("❌ Aturan 4 Gagal: Nilai ketersediaan melanggar batas kalender (>365 hari).")
    failed_rules += 1
else:
    print("✅ Aturan 4 Lulus: Batas ketersediaan hari masuk akal.")

# Aturan 5: Kelengkapan data wilayah
res5 = pd.read_sql_query("SELECT COUNT(*) as cacat FROM fact_airbnb WHERE city IS NULL OR city = '';", conn_elt)
if res5['cacat'].iloc[0] > 0:
    print("❌ Aturan 5 Gagal: Terdapat data sewa yang kehilangan nama kota.")
    failed_rules += 1
else:
    print("✅ Aturan 5 Lulus: Informasi spasial kota lengkap.")

# Aturan 6: Kepemilikan listings host
res6 = pd.read_sql_query("SELECT COUNT(*) as cacat FROM dim_host WHERE calculated_host_listings_count < 1;", conn_elt)
if res6['cacat'].iloc[0] > 0:
    print("❌ Aturan 6 Gagal: Aset hitung kepemilikan host tidak valid (<1).")
    failed_rules += 1
else:
    print("✅ Aturan 6 Lulus: Validasi kepemilikan aset host sukses.")

print("-" * 60)
if failed_rules == 0:
    print("🎉 PIPELINE ELT VALID! Gudang data siap disajikan untuk pelaporan bisnis.")
else:
    print(f"⚠ Perhatian: Ditemukan {failed_rules} kegagalan pada arsitektur ELT.")


# =============================================================
# PART B: EKSEKUSI 8 QUERY ANALITIK BISNIS
# =============================================================
# Query 1: Total volume data warehouse setelah transform via SQL
jalankan_analisis_sql(
    1, "Total Transaksi Valid Terisi pada Tabel Fakta",
    "SELECT COUNT(*) AS total_records_warehouse FROM fact_airbnb;"
)

# Query 2: Rata-rata harga asli per segmen ekonomi (Fitur Turunan 1)
jalankan_analisis_sql(
    2, "Rata-rata Harga Asli Berdasarkan Kelompok Kelas Ekonomi",
    """SELECT price_category, COUNT(*) as jumlah_properti, ROUND(AVG(price), 2) as rata_rata_harga
       FROM fact_airbnb
       GROUP BY price_category;"""
)

# Query 3: Top 5 Hub Bisnis Airbnb dengan performa omzet tertinggi (Fitur Turunan 4)
jalankan_analisis_sql(
    3, "Top 5 Kota dengan Akumulasi Estimasi Pendapatan Kotor Terbesar",
    """SELECT city, SUM(total_estimated_revenue) as total_pendapatan_kota
       FROM fact_airbnb
       GROUP BY city
       ORDER BY total_pendapatan_kota DESC
       LIMIT 5;"""
)

# Query 4: Pengaruh dominasi host bisnis terhadap akumulasi umpan balik konsumen (Fitur Turunan 5)
jalankan_analisis_sql(
    4, "Analisis Keterlibatan Review Berdasarkan Skala Bisnis Kepemilikan Host",
    """SELECT h.host_dominance, COUNT(f.id) as volume_listing, SUM(f.number_of_reviews) as total_ulasan
       FROM fact_airbnb f
       JOIN dim_host h ON f.host_id = h.host_id
       GROUP BY h.host_dominance;"""
)

# Query 5: Distribusi room_type beserta tingkat kesiapan sewa jangka panjang (Fitur Turunan 3)
jalankan_analisis_sql(
    5, "Karakteristik Tipe Akomodasi & Rasio Siap Sewa Tahunan",
    """SELECT room_type, COUNT(*) as total_listing, ROUND(AVG(price), 2) as rata_rata_harga, SUM(is_highly_available) as kuota_siap_tahunan
       FROM fact_airbnb
       GROUP BY room_type;"""
)

# Query 6: Tren rata-rata harga tahunan (Menggunakan Relasi Dimensi Waktu)
jalankan_analisis_sql(
    6, "Tren Fluktuasi Nilai Sewa Rata-rata Berdasarkan Tahun Review",
    """SELECT t.year, COUNT(f.id) as jumlah_transaksi, ROUND(AVG(f.price), 2) as rata_rata_harga_tahunan
       FROM fact_airbnb f
       JOIN dim_time t ON f.last_review = t.last_review
       GROUP BY t.year
       ORDER BY t.year ASC;"""
)

# Query 7: Penggabungan data internal dengan data sekunder hasil tangkapan API (Enrichment)
jalankan_analisis_sql(
    7, "Korelasi Populasi Kota (Data Mentah API) dengan Kerapatan Listing Properti",
    """SELECT f.city, c.populasi_kota_api, COUNT(f.id) as kerapatan_properti
       FROM fact_airbnb f
       JOIN dim_city c ON f.city = c.city
       WHERE c.populasi_kota_api IS NOT NULL
       GROUP BY f.city, c.populasi_kota_api
       ORDER BY kerapatan_properti DESC
       LIMIT 5;"""
)

# Query 8: Rasio sewa jangka panjang produk ternormalisasi
jalankan_analisis_sql(
    8, "Persentase Properti dengan Ketersediaan Tinggi per Kategori Harga",
    """SELECT price_category, COUNT(*) as total_unit, SUM(is_highly_available) as unit_tersedia_tinggi,
       ROUND((CAST(SUM(is_highly_available) AS FLOAT) / COUNT(*)) * 100, 2) || '%' as rasio_persentase
       FROM fact_airbnb
       GROUP BY price_category;"""
)

# Tutup koneksi secara aman
conn_elt.close()
print("\n🎉 VALIDASI KUALITAS & 8 QUERY UTAMA PIPELINE ELT SELESAI 100%!")

--- [ELT PHASE: VALIDATION & ANALYTICS VIA SQL] ---

🔍 Memulai 6 Aturan Uji Validasi Kualitas Data Warehouse...
✅ Aturan 1 Lulus: Seluruh ID transaksi 100% unik.
✅ Aturan 2 Lulus: Logika harga valid (tidak ada minus).
✅ Aturan 3 Lulus: Tipe data kolom minimum_nights konsisten berupa angka.
✅ Aturan 4 Lulus: Batas ketersediaan hari masuk akal.
✅ Aturan 5 Lulus: Informasi spasial kota lengkap.
✅ Aturan 6 Lulus: Validasi kepemilikan aset host sukses.
------------------------------------------------------------
🎉 PIPELINE ELT VALID! Gudang data siap disajikan untuk pelaporan bisnis.

📊 QUERY 1: Total Transaksi Valid Terisi pada Tabel Fakta


,total_records_warehouse
0,212637



📊 QUERY 2: Rata-rata Harga Asli Berdasarkan Kelompok Kelas Ekonomi


,price_category,jumlah_properti,rata_rata_harga
0,Mahal,71344,278.23
1,Murah,46499,57.75
2,Sedang,94794,125.98



📊 QUERY 3: Top 5 Kota dengan Akumulasi Estimasi Pendapatan Kotor Terbesar


...


📊 QUERY 7: Korelasi Populasi Kota (Data Mentah API) dengan Kerapatan Listing Properti


,city,populasi_kota_api,kerapatan_properti



📊 QUERY 8: Persentase Properti dengan Ketersediaan Tinggi per Kategori Harga


,price_category,total_unit,unit_tersedia_tinggi,rasio_persentase
0,Mahal,71344,36984,51.84%
1,Murah,46499,19115,41.11%
2,Sedang,94794,44910,47.38%



🎉 VALIDASI KUALITAS & 8 QUERY UTAMA PIPELINE ELT SELESAI 100%!


### Pemisahan Database

In [4]:
import sqlite3
import pandas as pd

# 1. Hubungkan ke database ELT Anda
db_path_elt = 'data_warehouse_elt/airbnb_elt_warehouse.db'
conn_elt = sqlite3.connect(db_path_elt)

# 2. Ambil data dari Tabel Fakta Utama (fact_airbnb)
df_fact_elt = pd.read_sql_query("SELECT * FROM fact_airbnb;", conn_elt)

# 3. Simpan menjadi file CSV di folder utama Colab Anda
df_fact_elt.to_csv('/content/fact_airbnb_elt.csv', index=False)

conn_elt.close()
print("✅ Sukses! File 'fact_airbnb_elt.csv' berhasil diciptakan dan siap Anda download.")

✅ Sukses! File 'fact_airbnb_elt.csv' berhasil diciptakan dan siap Anda download.


### Up to Date Data

In [5]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import pandas as pd
import sqlite3
import time

print("--- [AUTOMATION: LIVE GOOGLE SHEETS SYNC] ---")
print("🔄 Menghubungkan ke Google Sheets Anda...")
creds, _ = default()
gc = gspread.authorize(creds)

# 1. Ambil data fakta terbaru dari database SQL
db_path_elt = 'data_warehouse_elt/airbnb_elt_warehouse.db'
conn_elt = sqlite3.connect(db_path_elt)
df_fact = pd.read_sql_query("SELECT * FROM fact_airbnb;", conn_elt)
conn_elt.close()

# Mengisi nilai kosong dengan string agar tidak error saat konversi JSON
df_fact = df_fact.fillna('')

# 2. Membuat file Google Sheets baru di Drive Anda
print("🔄 Menciptakan Spreadsheet baru 'Live_Data_Warehouse_Airbnb'...")
sh = gc.create('Live_Data_Warehouse_Airbnb')
worksheet = sh.get_worksheet(0)

# =========================================================================
# SOLUSI UTAMA: Perluas ukuran grid Google Sheets di awal agar muat semua data
# =========================================================================
total_rows_needed = len(df_fact) + 100
total_cols_needed = len(df_fact.columns)
print(f"📐 Memperluas ukuran grid Google Sheets menjadi {total_rows_needed} baris...")
worksheet.resize(rows=total_rows_needed, cols=total_cols_needed)

# Menyiapkan seluruh data (Header + Baris Data)
header = df_fact.columns.values.tolist()
rows = df_fact.values.tolist()

# 3. Proses Pengiriman Secara Bertahap (Chunking)
print(f"🚀 Memulai pengiriman data secara bertahap ({len(df_fact)} baris)...")

# Menggunakan named arguments (range_name=, values=) untuk menghilangkan warning kuning
worksheet.update(range_name='A1', values=[header])

chunk_size = 20000
total_rows = len(rows)

for start_idx in range(0, total_rows, chunk_size):
    end_idx = min(start_idx + chunk_size, total_rows)
    chunk_data = rows[start_idx:end_idx]

    # Menghitung posisi baris di Google Sheets
    range_start_row = start_idx + 2
    target_cell = f"A{range_start_row}"

    print(f"📥 Menyuntikkan baris data {start_idx + 1} sampai {end_idx} ke Google Sheets...")
    worksheet.update(range_name=target_cell, values=chunk_data)

    # Beri jeda 1 detik tiap cicilan agar tidak terkena Rate Limit Google API
    time.sleep(1)

print(f"\n🎉 SUKSES BESAR! File 'Live_Data_Warehouse_Airbnb' kini sudah terisi penuh 100% di Google Drive Anda secara aman!")

--- [AUTOMATION: LIVE GOOGLE SHEETS SYNC] ---
🔄 Menghubungkan ke Google Sheets Anda...
🔄 Menciptakan Spreadsheet baru 'Live_Data_Warehouse_Airbnb'...
📐 Memperluas ukuran grid Google Sheets menjadi 212737 baris...
🚀 Memulai pengiriman data secara bertahap (212637 baris)...
📥 Menyuntikkan baris data 1 sampai 20000 ke Google Sheets...
📥 Menyuntikkan baris data 20001 sampai 40000 ke Google Sheets...
📥 Menyuntikkan baris data 40001 sampai 60000 ke Google Sheets...
📥 Menyuntikkan baris data 60001 sampai 80000 ke Google Sheets...
📥 Menyuntikkan baris data 80001 sampai 100000 ke Google Sheets...
📥 Menyuntikkan baris data 100001 sampai 120000 ke Google Sheets...
📥 Menyuntikkan baris data 120001 sampai 140000 ke Google Sheets...
📥 Menyuntikkan baris data 140001 sampai 160000 ke Google Sheets...
📥 Menyuntikkan baris data 160001 sampai 180000 ke Google Sheets...
📥 Menyuntikkan baris data 180001 sampai 200000 ke Google Sheets...
📥 Menyuntikkan baris data 200001 sampai 212637 ke Google Sheets...

🎉 